# Statistical analysis of the composition geometry

We 


In [12]:
import json
from pathlib import Path
import pandas as pd
import numpy as np

REPO_ROOT = Path.cwd().parents[1]

In [13]:
# Load data
with open(REPO_ROOT / "results/composition/v2_phase125_normTrue_a4.5/scoring/summary.json") as f:
    summary = json.load(f)
  
# Metadata  
print(summary["model"], summary["layer"], summary["alpha"], summary["tau_value"])

# Per-composition entries
pairs = summary["pairs"]
print(pairs[0].keys)

# Extract per-composition data
rows = []
for p in pairs:
    if p.get("status") != "ok":
        continue
    rows.append({
        "trait_a": p["trait_a"], "trait_b": p["trait_b"],
        "cos": p["cos"], "regime": p["regime"],
        "comp_base": p["baseline"]["composition_mean"],
        "comp_steered": p["steered"]["composition_mean"],
        "coh_steered": p["steered"]["coherence_mean"],
        "delta_a_joint":  p["delta"]["trait_a_joint"],
        "delta_a_single": p["delta"]["trait_a_single"],
        "delta_b_joint":  p["delta"]["trait_b_joint"],
        "delta_b_single": p["delta"]["trait_b_single"],
        "delta_comp": p["delta"]["composition"],
        "delta_coh":  p["delta"]["coherence"],
    })
    
# Convert to pd dataframe
df = pd.DataFrame(rows)
df["regime"].value_counts()

meta-llama/Llama-3.1-8B-Instruct 17 4.5 0.8296650044918062
<built-in method keys of dict object at 0x12531d500>


regime
mixed          14
additive        7
suppressive     4
dominant        3
Name: count, dtype: int64

In [14]:
# Add semantic similarity
with open(REPO_ROOT / "results/semantic_similarity.json") as f:
    sem = json.load(f)
    
sem_lookup = {
    tuple(sorted([p["trait_a"], p["trait_b"]])): p["sem_sim"]
    for p in sem["pairs"]
}

df["sem_sim"] = df.apply(
    lambda r: sem_lookup[tuple(sorted([r["trait_a"], r["trait_b"]]))],
    axis=1,
)

In [15]:
# Look at dataframe
df.head(n=40)

,trait_a,trait_b,cos,regime,comp_base,comp_steered,coh_steered,delta_a_joint,delta_a_single,delta_b_joint,delta_b_single,delta_comp,delta_coh,sem_sim
0,apathetic,confidence,0.0143,dominant,33.72,52.65,79.80,33.25,41.36,4.61,21.87,18.93,-18.37,0.240104
1,apathetic,evil,0.2578,dominant,3.97,40.03,50.35,58.34,45.50,13.79,63.48,36.06,-46.93,0.321475
2,apathetic,formality,0.0150,mixed,45.47,69.47,89.00,44.98,30.64,3.00,6.18,23.99,-8.84,0.246348
3,apathetic,hallucinating,-0.1642,additive,7.77,43.88,69.54,28.98,38.16,43.24,47.25,36.11,-23.55,0.198113
4,apathetic,humorous,0.1923,mixed,1.35,53.79,49.15,60.36,30.84,44.51,78.57,52.44,-48.05,0.232589
5,apathetic,impolite,0.6948,mixed,1.09,54.43,55.92,55.51,35.12,51.16,72.44,53.33,-40.70,0.469881
6,apathetic,sycophantic,0.0503,mixed,4.39,34.49,65.50,35.79,42.21,24.42,79.85,30.10,-32.28,0.344080
7,confidence,evil,0.2294,mixed,28.32,48.14,63.91,9.52,25.28,30.13,53.33,19.82,-32.45,0.209184
8,confidence,formality,0.2262,mixed,77.59,85.44,92.13,12.28,18.80,3.42,4.29,7.85,-5.50,0.398346
9,confidence,hallucinating,0.2700,additive,26.35,56.93,81.24,21.42,23.73,38.87,37.53,30.59,-11.37,0.251901


### Cross-tab for cosine v. regime

We perform a descriptive analysis to see whether simple cross-tabulation predicts a regime using cosine similarity (Gram) matrix $G$.

In [16]:
# Add column with absolute value of cosine similarities
df['cosine_abs'] = df['cos'].abs()

# Bin cosine with the same strata of the previous analysis
df['cos_bin'] = pd.cut(
    df['cosine_abs'],
    bins=[0, 0.2, 0.5, 1.0],
    labels=['low', 'moderate', 'high']
)

df.head(n=40)

,trait_a,trait_b,cos,regime,comp_base,comp_steered,coh_steered,delta_a_joint,delta_a_single,delta_b_joint,delta_b_single,delta_comp,delta_coh,sem_sim,cosine_abs,cos_bin
0,apathetic,confidence,0.0143,dominant,33.72,52.65,79.80,33.25,41.36,4.61,21.87,18.93,-18.37,0.240104,0.0143,low
1,apathetic,evil,0.2578,dominant,3.97,40.03,50.35,58.34,45.50,13.79,63.48,36.06,-46.93,0.321475,0.2578,moderate
2,apathetic,formality,0.0150,mixed,45.47,69.47,89.00,44.98,30.64,3.00,6.18,23.99,-8.84,0.246348,0.0150,low
3,apathetic,hallucinating,-0.1642,additive,7.77,43.88,69.54,28.98,38.16,43.24,47.25,36.11,-23.55,0.198113,0.1642,low
4,apathetic,humorous,0.1923,mixed,1.35,53.79,49.15,60.36,30.84,44.51,78.57,52.44,-48.05,0.232589,0.1923,low
5,apathetic,impolite,0.6948,mixed,1.09,54.43,55.92,55.51,35.12,51.16,72.44,53.33,-40.70,0.469881,0.6948,high
6,apathetic,sycophantic,0.0503,mixed,4.39,34.49,65.50,35.79,42.21,24.42,79.85,30.10,-32.28,0.344080,0.0503,low
7,confidence,evil,0.2294,mixed,28.32,48.14,63.91,9.52,25.28,30.13,53.33,19.82,-32.45,0.209184,0.2294,moderate
8,confidence,formality,0.2262,mixed,77.59,85.44,92.13,12.28,18.80,3.42,4.29,7.85,-5.50,0.398346,0.2262,moderate
9,confidence,hallucinating,0.2700,additive,26.35,56.93,81.24,21.42,23.73,38.87,37.53,30.59,-11.37,0.251901,0.2700,moderate


In [17]:
# Cross-tab
pd.crosstab(df['cos_bin'], df['regime'])

regime,additive,dominant,mixed,suppressive
cos_bin,,,,
low,2,1,7,2
moderate,5,2,6,1
high,0,0,1,1


In [19]:
# Small-sample association test on the cos_bin x regime cross-tab.
# At n=28 with ~10/12 cells having expected count < 5, the chi-square approximation
# is invalid. scipy's fisher_exact supports only 2x2 tables, so we use two forms:
#   (1) full r x c table  -> Monte-Carlo "exact" test (no expected-count assumption)
#   (2) interpretable 2x2 -> a genuine Fisher's exact test
from scipy.stats import fisher_exact, chi2_contingency

ct = pd.crosstab(df['cos_bin'], df['regime'])
print("Cross-tab (cos_bin x regime):")
print(ct, "\n")

# --- (1) Full table: Monte-Carlo exact test ---------------------------------
# Same hypothesis as the chi-square ("does the cosine bin associate with regime?"),
# but the null is built by permuting regime labels rather than assuming the
# asymptotic chi-square distribution. Valid for sparse tables.
sub      = df.dropna(subset=['cos_bin'])
bin_code = pd.Categorical(sub['cos_bin']).codes
reg_code = pd.Categorical(sub['regime']).codes
n_bin, n_reg = bin_code.max() + 1, reg_code.max() + 1

def _chi2_stat(b, r):
    t = np.zeros((n_bin, n_reg))
    np.add.at(t, (b, r), 1)
    exp = t.sum(1, keepdims=True) @ t.sum(0, keepdims=True) / t.sum()
    with np.errstate(divide='ignore', invalid='ignore'):
        return np.nansum((t - exp) ** 2 / exp)

rng_mc   = np.random.default_rng(0)
N_MC     = 10000
obs_chi2 = _chi2_stat(bin_code, reg_code)
perm     = np.fromiter((_chi2_stat(bin_code, rng_mc.permutation(reg_code))
                        for _ in range(N_MC)), float, N_MC)
p_mc     = (np.sum(perm >= obs_chi2) + 1) / (N_MC + 1)
print(f"(1) Full {ct.shape[0]}x{ct.shape[1]} table - Monte-Carlo exact test "
      f"({N_MC} permutations):")
print(f"    observed chi2 = {obs_chi2:.3f},  p = {p_mc:.4f}\n")

# --- (2) Interpretable 2x2 Fisher's exact -----------------------------------
# Collapse to the binary question the rest of the notebook cares about:
# near-orthogonal vs aligned  x  additive vs not.
df['cos_hi']      = (df['cosine_abs'] >= 0.2).astype(int)   # 0 = |cos|<0.2, 1 = >=0.2
additive          = (df['regime'] == 'additive').astype(int)
tab2              = pd.crosstab(df['cos_hi'], additive)
tab2.index.name   = '|cos|>=0.2'
tab2.columns.name = 'is_additive'
odds, p_fisher    = fisher_exact(tab2, alternative='two-sided')
print("(2) 2x2 Fisher's exact (|cos|>=0.2  x  is_additive):")
print(tab2)
print(f"    odds ratio = {odds:.3f},  p = {p_fisher:.4f}")


Cross-tab (cos_bin x regime):
regime    additive  dominant  mixed  suppressive
cos_bin                                         
low              2         1      7            2
moderate         5         2      6            1
high             0         0      1            1 

(1) Full 3x4 table - Monte-Carlo exact test (10000 permutations):
    observed chi2 = 4.563,  p = 0.6672

(2) 2x2 Fisher's exact (|cos|>=0.2  x  is_additive):
is_additive   0  1
|cos|>=0.2        
0            10  2
1            11  5
    odds ratio = 2.273,  p = 0.6618


### Correlation between cosine and semantic similarities
We compute Pearson correlation between absolute cosine similarity and semantic similarity, to understand if the latter could be a confound.

In [22]:
from scipy.stats import pearsonr

# Confound check (single source of truth): is semantic similarity collinear with |cos|?
# r_cs / p_cs are reused by the partial-Spearman motivation and the RQ1 summary save.
r_cs, p_cs = pearsonr(df["cosine_abs"], df["sem_sim"])
print(f"Pearson r(cosine_abs, sem_sim) = {r_cs:+.3f}  (p = {p_cs:.4f})")
print(df[["cosine_abs", "sem_sim"]].describe().round(3))

Pearson r(cosine_abs, sem_sim) = +0.532
       cosine_abs  sem_sim
count      28.000   28.000
mean        0.242    0.276
std         0.162    0.077
min         0.014    0.185
25%         0.120    0.222
50%         0.230    0.245
75%         0.358    0.322
max         0.695    0.470


With a correlation over +0.5, we should account for similarity as a confound for our analysis when using absolute cosine similarity.
On the other hand, semantic similarity should not be a confounder for signed cosine similarity, since the former is always positive.

In [23]:
corr = df[["cos", "sem_sim"]].corr().iloc[0, 1]
print(f"Pearson r(cos, sem_sim) = {corr:+.3f}")
print(df[["cos", "sem_sim"]].describe().round(3))

Pearson r(cos, sem_sim) = +0.135
          cos  sem_sim
count  28.000   28.000
mean    0.151    0.276
std     0.252    0.077
min    -0.522    0.185
25%    -0.002    0.222
50%     0.209    0.245
75%     0.305    0.322
max     0.695    0.470


## Binary logistic regression

We run **four binary logistic regressions** (predicting `is_additive`) and, in a separate section below, **two multinomial regressions** (predicting the 4-class regime).

Experiments 1–2 test $|\cos|$ and semantic similarity in isolation; Exp 3 puts them together as a confound control (if $|\cos|$ keeps a negative slope and LOO AUC improves over the semantic-only model, the geometric signal is real, not a semantic proxy); Exp 4 swaps in signed cosine to test whether direction adds information.

_We omit a signed-cos + sem model: signed cosine and semantic similarity are essentially uncorrelated here ($r\approx0.14$), so controlling for semantics would not change the signed-cos estimate._

| # | Outcome | Predictors |
|---|---------|------------|
| 1 | is\_additive | $\lvert\cos\rvert$ |
| 2 | is\_additive | semantic similarity |
| 3 | is\_additive | $\lvert\cos\rvert$ + sem\_sim |
| 4 | is\_additive | signed $\cos$ |

#### Metrics

For each model we report four quantities:

| Metric | How it is computed | What it tells you |
|--------|--------------------|-------------------|
| **In-sample AUC** | ROC-AUC on the same 28 pairs used to fit the model | Optimistic upper bound — confirms the model can fit the data; not a measure of generalisation |
| **LOO AUC** | Pool the 28 held-out probabilities from leave-one-out CV, then compute one AUC | **Primary metric.** The honest out-of-sample estimate of predictive signal |
| **Bootstrap 95% CI** | 2 000 stratified resamples, each with a fresh model refit; percentile interval on in-sample AUC | Uncertainty around the AUC given the small sample size (n = 28) |
| **Permutation p** | Fraction of 10 000 label-shuffled AUCs ≥ observed, one-sided | Whether the predictor explains more variance than chance |

**Why AUC and not accuracy?** With 7 additive pairs out of 28, a classifier that always predicts "non-additive" reaches 75 % accuracy but AUC = 0.5. AUC is threshold-free and handles class imbalance.

**What counts as evidence?** LOO AUC > 0.7 *and* permutation p < 0.05 together. LOO AUC is the primary metric; in-sample AUC is shown for reference only.

In [ ]:
# Save experiments results
results = []

In [ ]:
# Add binary variable for additive v. non-additive
df['is_additive'] = (df['regime'] == 'additive').astype(int)
df.head()

In [ ]:
from sklearn.base import clone
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score
from sklearn.model_selection import cross_val_predict, StratifiedKFold, LeaveOneOut

# Single rng instantiated
rng = np.random.default_rng(0)
N_BOOT  = 2000
N_PERM  = 10000

In [ ]:
def bootstrap_perm(model, X, y, exp_auc):

    def fit_auc(X_tr, y_tr, X_ev, y_ev):
        m = clone(model).fit(X_tr, y_tr)
        return roc_auc_score(y_ev, m.predict_proba(X_ev)[:, 1])

    pos_idx = np.where(y == 1)[0]
    neg_idx = np.where(y == 0)[0]
    n_obs   = len(y)

    # --- LOO slope stability ---
    # For each of the 36 leave-one-out folds: fit on n-1 points, extract the
    # slope coefficient (coef_[0, :]), track sign and range.
    # Also count folds where the training set has < 2 positives — with 3
    # positives total and LOO, this only fires when a positive is held out AND
    # there are only 2 remaining. Expected count: 0 (we have 3 positives so
    # the worst case is 2 in training), but the check is a useful sanity gate
    # for future experiments with fewer positives.
    loo_slopes        = []
    loo_degen_folds   = 0
    for i in range(n_obs):
        mask   = np.ones(n_obs, dtype=bool)
        mask[i] = False
        y_tr   = y[mask]
        if (y_tr == 1).sum() < 2:
            loo_degen_folds += 1
            continue
        try:
            m = clone(model).fit(X[mask], y_tr)
            loo_slopes.append(m.coef_[0].tolist())    # one value per feature
        except Exception:
            pass

    loo_slopes_arr = np.array(loo_slopes)              # shape: (n_valid_folds, n_features)
    print(f"LOO folds with < 2 positives in training: {loo_degen_folds}")
    for feat_idx in range(loo_slopes_arr.shape[1]):
        col = loo_slopes_arr[:, feat_idx]
        print(f"  feature {feat_idx}: slope min={col.min():+.3f}  "
              f"max={col.max():+.3f}  "
              f"n_negative={int((col < 0).sum())}/{len(col)}")

    # --- Bootstrap 95% CI (stratified, evaluate on resample) ---
    boot_aucs = []
    for _ in range(N_BOOT):
        pi  = rng.choice(pos_idx, size=len(pos_idx), replace=True)
        ni  = rng.choice(neg_idx, size=len(neg_idx), replace=True)
        idx = np.concatenate([pi, ni])
        if len(np.unique(y[idx])) < 2:
            continue
        try:
            boot_aucs.append(fit_auc(X[idx], y[idx], X[idx], y[idx]))
        except ValueError:
            pass

    boot_aucs = np.array(boot_aucs)
    ci_lo, ci_hi = np.percentile(boot_aucs, [2.5, 97.5])
    print(f"AUC = {exp_auc:.3f}  95% CI [{ci_lo:.3f}, {ci_hi:.3f}]  "
          f"(n_boot={len(boot_aucs)}  shape: min={boot_aucs.min():.3f} "
          f"median={np.median(boot_aucs):.3f} max={boot_aucs.max():.3f})")

    # --- Permutation p-value (one-sided) ---
    perm_aucs = []
    for _ in range(N_PERM):
        y_perm = rng.permutation(y)
        perm_aucs.append(fit_auc(X, y_perm, X, y_perm))

    perm_aucs = np.array(perm_aucs)
    p_value   = (np.sum(perm_aucs >= exp_auc) + 1) / (N_PERM + 1)
    print(f"permutation p (one-sided) = {p_value:.4f}")

    return {
        "auc_insample":           exp_auc,
        "ci_lo":                  float(ci_lo),
        "ci_hi":                  float(ci_hi),
        "n_boot":                 len(boot_aucs),
        "boot_aucs":              boot_aucs.tolist(),      # full distribution for figures
        "perm_p":                 float(p_value),
        "perm_auc_mean":          float(perm_aucs.mean()),
        "loo_slope_min":          float(loo_slopes_arr[:, 0].min()),
        "loo_slope_max":          float(loo_slopes_arr[:, 0].max()),
        "loo_slope_n_neg":        int((loo_slopes_arr[:, 0] < 0).sum()),
        "loo_slope_n_folds":      len(loo_slopes),
        "loo_degen_folds":        loo_degen_folds,
    }

In [ ]:
scaler = StandardScaler()
binary_model = LogisticRegression(penalty=None)
multi_model = LogisticRegression(penalty=None, multi_class='multinomial')

### Exp 1 — Binary LR: $|\cos|$ → is\_additive

Baseline geometric model. Tests whether pairs with more aligned steering vectors are *less* likely to compose additively, as the Park LRH predicts.

In [ ]:
# Load exp 1 data
X_abs   = df[['cosine_abs']].to_numpy()
X_abs_s = scaler.fit_transform(X_abs)
y_add   = df["is_additive"].to_numpy()

# Fit on all data for in-sample AUC + coefficient sign
exp1_results = binary_model.fit(X_abs_s, y_add)
y_proba_in   = exp1_results.predict_proba(X_abs_s)[:, 1]
exp1_auc     = roc_auc_score(y_add, y_proba_in)

# Leave-one-out: fit on n-1, predict held-out probability, pool 36 probabilities, single AUC
loo            = LeaveOneOut()
y_proba_loo    = cross_val_predict(binary_model, X_abs_s, y_add, cv=loo, method="predict_proba")[:, 1]
exp1_auc_loo   = roc_auc_score(y_add, y_proba_loo)

print(f"in-sample AUC      = {exp1_auc:.3f}")
print(f"LOO (pooled) AUC   = {exp1_auc_loo:.3f}")
print(f"slope on cosine_abs (scaled) = {exp1_results.coef_[0,0]:+.3f}  "
      f"(negative => high |cos| reduces P(additive) — the geometric prediction)")

In [ ]:
# Run CI bootstrap and p-value permutation
stats = bootstrap_perm(binary_model, X_abs_s, y_add, exp1_auc)
results.append({
    "experiment":  "B1_abs_cos",
    "outcome":     "is_additive",
    "predictors":  "cosine_abs",
    "n_features":  1,
    "auc_loo":     exp1_auc_loo,
    "slope_abs_cos": float(exp1_results.coef_[0, 0]),
    "slope_both_antisocial": None,         # not in this model
    "slope_signed_cos":      None,
    **stats,
})

### Exp 2 — Binary LR: semantic similarity → is\_additive

Semantic baseline. Checks whether trait relatedness in embedding space (not geometry) already explains additivity, before introducing the steering-vector cosine.

In [ ]:
# Load exp 2 data
X_sem   = df[['sem_sim']].to_numpy()
X_sem_s = scaler.fit_transform(X_sem)
y_add   = df["is_additive"].to_numpy()

# Fit on all data for in-sample AUC + coefficient sign
exp2_results = binary_model.fit(X_sem_s, y_add)
y_proba_in   = exp2_results.predict_proba(X_sem_s)[:, 1]
exp2_auc     = roc_auc_score(y_add, y_proba_in)

# Leave-one-out: fit on n-1, predict held-out probability, pool 36 probabilities, single AUC
loo            = LeaveOneOut()
y_proba_loo    = cross_val_predict(binary_model, X_sem_s, y_add, cv=loo, method="predict_proba")[:, 1]
exp2_auc_loo   = roc_auc_score(y_add, y_proba_loo)

print(f"in-sample AUC      = {exp2_auc:.3f}")
print(f"LOO (pooled) AUC   = {exp2_auc_loo:.3f}")
print(f"slope on semantic_similarity (scaled) = {exp2_results.coef_[0,0]:+.3f}  "
      f"(negative => high |sem_sim| reduces P(additive) — the geometric prediction)")

In [ ]:
# Run CI bootstrap and p-value permutation
stats = bootstrap_perm(binary_model, X_sem_s, y_add, exp2_auc)
results.append({
    "experiment":  "B2_sem_sim",
    "outcome":     "is_additive",
    "predictors":  "semantic_similarity",
    "n_features":  1,
    "auc_loo":     exp2_auc_loo,
    "slope_sem_sim": float(exp2_results.coef_[0, 0]),
    "slope_both_antisocial": None,         # not in this model
    "slope_signed_cos":      None,
    **stats,
})

### Exp 3 — Binary LR: $|\cos|$ + sem\_sim → is\_additive

Confound control. Both predictors in the same model. If $|\cos|$ retains a negative slope and LOO AUC improves over Exp 2, the geometric signal is real and not merely tracking semantic similarity.

In [ ]:
X_both_5   = df[['cosine_abs', 'sem_sim']].to_numpy()
X_both_s_5 = scaler.fit_transform(X_both_5)

exp5_results = binary_model.fit(X_both_s_5, y_add)
y_proba_in_5 = exp5_results.predict_proba(X_both_s_5)[:, 1]
exp5_auc     = roc_auc_score(y_add, y_proba_in_5)

loo           = LeaveOneOut()
y_proba_loo_5 = cross_val_predict(binary_model, X_both_s_5, y_add, cv=loo, method="predict_proba")[:, 1]
exp5_auc_loo  = roc_auc_score(y_add, y_proba_loo_5)

print(f"in-sample AUC      = {exp5_auc:.3f}")
print(f"LOO (pooled) AUC   = {exp5_auc_loo:.3f}")
print(f"slope on cosine_abs (scaled) = {exp5_results.coef_[0,0]:+.3f}")
print(f"slope on sem_sim   (scaled)  = {exp5_results.coef_[0,1]:+.3f}")

In [ ]:
stats = bootstrap_perm(binary_model, X_both_s_5, y_add, exp5_auc)
results.append({
    "experiment":  "B3_abs_cos_sem",
    "outcome":     "is_additive",
    "predictors":  "cosine_abs + sem_sim",
    "n_features":  2,
    "auc_loo":     exp5_auc_loo,
    "slope_abs_cos":         float(exp5_results.coef_[0, 0]),
    "slope_both_antisocial": None,
    "slope_signed_cos":      None,
    "slope_sem_sim":         float(exp5_results.coef_[0, 1]),
    **stats,
})

### Exp 4 — Binary LR: signed $\cos$ → is\_additive

Sign sensitivity check. If additivity depends only on the magnitude of interference, signed cos should not outperform $|\cos|$. A higher LOO AUC here would suggest direction matters too.

In [ ]:
X_sgn   = df[['cos']].to_numpy()
X_sgn_s = scaler.fit_transform(X_sgn)
y_add   = df["is_additive"].to_numpy()

exp3_results = binary_model.fit(X_sgn_s, y_add)
y_proba_in_3 = exp3_results.predict_proba(X_sgn_s)[:, 1]
exp3_auc     = roc_auc_score(y_add, y_proba_in_3)

loo           = LeaveOneOut()
y_proba_loo_3 = cross_val_predict(binary_model, X_sgn_s, y_add, cv=loo, method="predict_proba")[:, 1]
exp3_auc_loo  = roc_auc_score(y_add, y_proba_loo_3)

print(f"in-sample AUC      = {exp3_auc:.3f}")
print(f"LOO (pooled) AUC   = {exp3_auc_loo:.3f}")
print(f"slope on signed_cos (scaled) = {exp3_results.coef_[0,0]:+.3f}  "
      f"(negative => high signed_cos reduces P(additive))")

In [ ]:
stats = bootstrap_perm(binary_model, X_sgn_s, y_add, exp3_auc)
results.append({
    "experiment":  "B4_signed_cos",
    "outcome":     "is_additive",
    "predictors":  "signed_cos",
    "n_features":  1,
    "auc_loo":     exp3_auc_loo,
    "slope_abs_cos":         None,
    "slope_both_antisocial": None,
    "slope_signed_cos":      float(exp3_results.coef_[0, 0]),
    **stats,
})

## Multinomial logistic regression

Same predictors, but predicting the full 4-class `regime` (additive / dominant / mixed / suppressive) instead of the binary additive label. AUC is macro one-vs-rest. With n = 28 across 4 classes this is heavily underpowered. This is exploratory only.

| # | Outcome | Predictors |
|---|---------|------------|
| 1 | regime | $\lvert\cos\rvert$ |
| 2 | regime | $\lvert\cos\rvert$ + sem\_sim |

In [ ]:
def bootstrap_perm_multi(model, X, y, exp_auc):
    classes = sorted(np.unique(y))
    n_obs   = len(y)
    n_feat  = X.shape[1]

    def fit_auc_multi(X_tr, y_tr, X_ev, y_ev):
        if len(np.unique(y_tr)) < len(classes):
            return None
        m = clone(model).fit(X_tr, y_tr)
        if len(m.classes_) < len(classes):
            return None
        return roc_auc_score(y_ev, m.predict_proba(X_ev), multi_class='ovr', average='macro')

    loo_slopes      = []
    loo_degen_folds = 0
    for i in range(n_obs):
        mask = np.ones(n_obs, dtype=bool)
        mask[i] = False
        y_tr = y[mask]
        if len(np.unique(y_tr)) < len(classes):
            loo_degen_folds += 1
            continue
        try:
            m = clone(model).fit(X[mask], y_tr)
            loo_slopes.append(np.abs(m.coef_).mean(axis=0).tolist())
        except Exception:
            pass

    loo_slopes_arr = np.array(loo_slopes)
    print(f"LOO folds with missing class in training: {loo_degen_folds}")
    for feat_idx in range(n_feat):
        col = loo_slopes_arr[:, feat_idx]
        print(f"  feature {feat_idx}: mean|slope| min={col.min():.3f}  max={col.max():.3f}")

    boot_aucs = []
    for _ in range(N_BOOT):
        idx = rng.choice(n_obs, size=n_obs, replace=True)
        try:
            v = fit_auc_multi(X[idx], y[idx], X[idx], y[idx])
            if v is not None:
                boot_aucs.append(v)
        except Exception:
            pass

    boot_aucs = np.array(boot_aucs)
    ci_lo, ci_hi = np.percentile(boot_aucs, [2.5, 97.5])
    print(f"AUC = {exp_auc:.3f}  95% CI [{ci_lo:.3f}, {ci_hi:.3f}]  "
          f"(n_boot={len(boot_aucs)}  shape: min={boot_aucs.min():.3f} "
          f"median={np.median(boot_aucs):.3f} max={boot_aucs.max():.3f})")

    perm_aucs = []
    for _ in range(N_PERM):
        y_perm = rng.permutation(y)
        try:
            v = fit_auc_multi(X, y_perm, X, y_perm)
            if v is not None:
                perm_aucs.append(v)
        except Exception:
            pass

    perm_aucs = np.array(perm_aucs)
    p_value   = (np.sum(perm_aucs >= exp_auc) + 1) / (N_PERM + 1)
    print(f"permutation p (one-sided) = {p_value:.4f}")

    return {
        "auc_insample":    exp_auc,
        "ci_lo":           float(ci_lo),
        "ci_hi":           float(ci_hi),
        "n_boot":          len(boot_aucs),
        "boot_aucs":       boot_aucs.tolist(),
        "perm_p":          float(p_value),
        "perm_auc_mean":   float(perm_aucs.mean()),
        "loo_degen_folds": loo_degen_folds,
        "loo_n_folds":     len(loo_slopes),
    }

### Exp 1 — Multinomial LR: $|\cos|$ → regime

Extends the binary question to all four regimes (additive, dominant, suppressive, mixed). AUC is macro one-vs-rest. Note: with n = 28 and 4 classes, this model is heavily underpowered — treat as exploratory.

In [ ]:
y_multi   = df["regime"].to_numpy()
X_abs_s_4 = scaler.fit_transform(df[['cosine_abs']].to_numpy())

exp4_results = multi_model.fit(X_abs_s_4, y_multi)
probas_in_4  = exp4_results.predict_proba(X_abs_s_4)
exp4_auc     = roc_auc_score(y_multi, probas_in_4, multi_class='ovr', average='macro')

loo           = LeaveOneOut()
y_proba_loo_4 = cross_val_predict(multi_model, X_abs_s_4, y_multi, cv=loo, method="predict_proba")
exp4_auc_loo  = roc_auc_score(y_multi, y_proba_loo_4, multi_class='ovr', average='macro')

print(f"in-sample AUC (macro OVR) = {exp4_auc:.3f}")
print(f"LOO (macro OVR)           = {exp4_auc_loo:.3f}")
for i, c in enumerate(exp4_results.classes_):
    print(f"  class {c}: coef = {exp4_results.coef_[i, 0]:+.3f}")

In [ ]:
stats = bootstrap_perm_multi(multi_model, X_abs_s_4, y_multi, exp4_auc)
results.append({
    "experiment":  "M1_multi_abs_cos",
    "outcome":     "regime",
    "predictors":  "cosine_abs",
    "n_features":  1,
    "auc_loo":     exp4_auc_loo,
    **stats,
})

### Exp 2 — Multinomial LR: $|\cos|$ + sem\_sim → regime

Full multinomial model with both predictors. Exploratory: checks whether the two signals together can discriminate among regimes better than geometry alone (Exp 1).

In [ ]:
X_both_7   = df[['cosine_abs', 'sem_sim']].to_numpy()
X_both_s_7 = scaler.fit_transform(X_both_7)

exp7_results = multi_model.fit(X_both_s_7, y_multi)
probas_in_7  = exp7_results.predict_proba(X_both_s_7)
exp7_auc     = roc_auc_score(y_multi, probas_in_7, multi_class='ovr', average='macro')

loo           = LeaveOneOut()
y_proba_loo_7 = cross_val_predict(multi_model, X_both_s_7, y_multi, cv=loo, method="predict_proba")
exp7_auc_loo  = roc_auc_score(y_multi, y_proba_loo_7, multi_class='ovr', average='macro')

print(f"in-sample AUC (macro OVR) = {exp7_auc:.3f}")
print(f"LOO (macro OVR)           = {exp7_auc_loo:.3f}")
for i, c in enumerate(exp7_results.classes_):
    print(f"  class {c}: coef_abs={exp7_results.coef_[i,0]:+.3f}  coef_sem={exp7_results.coef_[i,1]:+.3f}")

In [ ]:
stats = bootstrap_perm_multi(multi_model, X_both_s_7, y_multi, exp7_auc)
results.append({
    "experiment":  "M2_multi_abs_sem",
    "outcome":     "regime",
    "predictors":  "cosine_abs + sem_sim",
    "n_features":  2,
    "auc_loo":     exp7_auc_loo,
    **stats,
})

In [ ]:
results_df = pd.DataFrame(results)
results_df

In [ ]:
print("Total pairs:", len(df))
print("\nRegime counts:")
print(df['regime'].value_counts())
print("\nBinary balance:")
print(df['is_additive'].value_counts())
print("\nMissing values:")
print(df[['cosine_abs', 'sem_sim', 'is_additive', 'cos', 'regime']].isna().sum())
print("\ncos distribution:")
print(df['cos'].describe())

## Continuous composition quality Q

$Q = \tfrac{1}{2}\!\left(\frac{\Delta_{a,\text{joint}}}{\max(\Delta_{a,\text{single}},\varepsilon)} + \frac{\Delta_{b,\text{joint}}}{\max(\Delta_{b,\text{single}},\varepsilon)}\right),\quad \varepsilon = 10^{-6}$

Each ratio is clipped to $[-2, 2]$ before averaging. $Q = 1$ means both axes were preserved exactly; $Q < 0$ means both axes were reversed.

In [130]:
CLIP = 2.0
EPS  = 1e-6

r_a_raw = df["delta_a_joint"] / df["delta_a_single"].clip(lower=EPS)
r_b_raw = df["delta_b_joint"] / df["delta_b_single"].clip(lower=EPS)

n_clip_a = int(((r_a_raw < -CLIP) | (r_a_raw > CLIP)).sum())
n_clip_b = int(((r_b_raw < -CLIP) | (r_b_raw > CLIP)).sum())
print(f"Ratios clipped to [{-CLIP}, {CLIP}]: r_a={n_clip_a}, r_b={n_clip_b}  (of {len(df)} pairs)")

df["Q"] = 0.5 * (r_a_raw.clip(-CLIP, CLIP) + r_b_raw.clip(-CLIP, CLIP))

print("\nQ distribution:")
print(df["Q"].describe().round(3))
print(f"\nQ >= 0.8: {(df['Q'] >= 0.8).sum()}")
print(f"Q >= 0.7: {(df['Q'] >= 0.7).sum()}")
print(f"Q >= 0.6: {(df['Q'] >= 0.6).sum()}")

print("\nSanity check — mean Q per regime (additive pairs should be near 1.0):")
print(df.groupby("regime")["Q"].agg(["mean", "min", "max"]).round(3))

additive_q = df.loc[df["regime"] == "additive", "Q"].mean()
note = "matches proposal" if additive_q > 0.70 else "LOWER THAN EXPECTED — check formula"
print(f"\nMean Q for additive pairs: {additive_q:.3f}  ({note})")

Ratios clipped to [-2.0, 2.0]: r_a=9, r_b=7  (of 36 pairs)

Q distribution:
count    36.000
mean      0.586
std       0.949
min      -0.995
25%      -0.439
50%       0.892
75%       1.250
max       1.995
Name: Q, dtype: float64

Q >= 0.8: 18
Q >= 0.7: 19
Q >= 0.6: 19

Sanity check — mean Q per regime (additive pairs should be near 1.0):
              mean    min    max
regime                          
additive    -0.518 -0.628 -0.416
dominant     0.130 -0.645  0.568
emergent     1.763  1.439  1.995
mixed        0.635 -0.995  1.539
suppressive -0.210 -0.821  0.176

Mean Q for additive pairs: -0.518  (LOWER THAN EXPECTED — check formula)


# Primary statistical tests (continuous)

The three bivariate predictor→Q tests (1–3) use the **same battery** — Spearman $\rho$ and Pearson
$r$, each with a two-sided permutation p (robust to `Q`'s clipping at n = 28) — so the predictors
are compared on equal footing. Test 4 adds the partial Spearman for `|cos|` (which is collinear
with sem, $r\approx0.53$); signed cos (test 2) needs no partialling ($\bot$ sem, $r\approx0.14$).
Test 5 is the parametric multiple-regression complement.

| # | Test | Target | Predictor(s) | What it checks |
|---|------|--------|--------------|----------------|
| 1 | Spearman + Pearson + perm | Q | $\lvert\cos\rvert$ | does geometric overlap *magnitude* track Q? |
| 2 | Spearman + Pearson + perm | Q | signed $\cos$ | does *signed* alignment track Q? (aligned → high Q, antipodal → low) |
| 3 | Spearman + Pearson + perm | Q | sem\_sim | semantic baseline (geometry-free) |
| 4 | Partial Spearman | Q | $\lvert\cos\rvert \mid$ sem\_sim | does the `|cos|` magnitude effect survive controlling for semantics? |
| 5 | OLS (multiple) | Q | $\lvert\cos\rvert$ + sem\_sim | partial slopes + $R^2$ (parametric complement to #4) |

**What counts as evidence?** Signed cos (#2) is the predictor that preserves the monotonic
aligned→high-Q / antipodal→low-Q relationship; `|cos|` (#1) collapses the sign and can null the
signal out, so a strong #2 with a weak #1 is itself informative. For `|cos|`, a significant
**partial Spearman (#4)** with low attenuation from #1 shows its effect is not a semantic proxy;
#5 reports the same as standardized $\beta$'s with bootstrap CIs.

_Caveat:_ under `normalize=True`, per-axis push $=\alpha\sqrt{(1+\cos)/2}$ is itself monotonic in
signed cos, so part of the signed-cos→Q relationship is built into the injection scheme; the
`peraxis` (v3) dataset disentangles geometry from injection design. Parametric p-values are treated
cautiously (`Q` is clipped to $[-2,2]$); permutation/bootstrap inference is preferred.


In [131]:
from scipy.stats import spearmanr, pearsonr

n = len(df)

def ols_resid(y, x):
    X = np.column_stack([np.ones(len(x)), x])
    beta = np.linalg.lstsq(X, y, rcond=None)[0]
    return y - X @ beta

def corr_battery(x, y, name, n_perm=N_PERM, seed=0):
    """Uniform bivariate battery for a predictor -> Q: Spearman rho and Pearson r,
    each with a two-sided permutation p (robust to Q's clipping at n=28). Applied
    identically to every predictor so they are compared on equal footing."""
    x, y = np.asarray(x, float), np.asarray(y, float)
    rho, p_rho = spearmanr(x, y)
    r,   p_r   = pearsonr(x, y)
    g = np.random.default_rng(seed)
    perm_rho = np.empty(n_perm)
    perm_r   = np.empty(n_perm)
    for i in range(n_perm):
        xp = g.permutation(x)
        perm_rho[i] = spearmanr(xp, y)[0]
        perm_r[i]   = pearsonr(xp, y)[0]
    p_rho_perm = (np.sum(np.abs(perm_rho) >= abs(rho)) + 1) / (n_perm + 1)
    p_r_perm   = (np.sum(np.abs(perm_r)   >= abs(r))   + 1) / (n_perm + 1)
    print(f"{name:>11s} -> Q:  "
          f"Spearman rho={rho:+.3f} (param p={p_rho:.3f}, perm p={p_rho_perm:.4f})   "
          f"Pearson r={r:+.3f} (param p={p_r:.3f}, perm p={p_r_perm:.4f})")
    return {"rho": float(rho), "p_rho": float(p_rho), "p_rho_perm": float(p_rho_perm),
            "r": float(r), "p_r": float(p_r), "p_r_perm": float(p_r_perm)}

# 1. |cos| -> Q
res_abs = corr_battery(df["cosine_abs"], df["Q"], "|cos|")
rho_cq, p_cq, r_cq, p_cq_perm = res_abs["rho"], res_abs["p_rho"], res_abs["r"], res_abs["p_r_perm"]

# 2. signed cos -> Q   (no partialling: signed cos is uncorrelated with sem, r~0.14)
res_signed = corr_battery(df["cos"], df["Q"], "signed cos")
rho_sc, p_sc, r_sc, p_sc_perm = res_signed["rho"], res_signed["p_rho"], res_signed["r"], res_signed["p_r_perm"]

# 3. sem_sim -> Q   (geometry-free baseline)
res_sem = corr_battery(df["sem_sim"], df["Q"], "sem_sim")
rho_sq, p_sq, r_sq, p_sq_perm = res_sem["rho"], res_sem["p_rho"], res_sem["r"], res_sem["p_r_perm"]

# 4. Partial Spearman: |cos| controlling sem_sim (only |cos| needs it -- signed cos is
#    uncorrelated with sem). Residualize Q and cosine_abs on sem_sim, then Spearman residuals.
resid_Q   = ols_resid(df["Q"].to_numpy(),          df["sem_sim"].to_numpy())
resid_cos = ols_resid(df["cosine_abs"].to_numpy(), df["sem_sim"].to_numpy())
rho_part, p_part = spearmanr(resid_cos, resid_Q)
attenuation_pct  = (rho_cq - rho_part) / abs(rho_cq) * 100 if rho_cq != 0 else float("nan")
print(f"\nPartial Spearman(|cos|, Q | sem_sim):  rho={rho_part:+.3f},  p={p_part:.3f}  "
      f"(raw rho={rho_cq:+.3f} -> partial {rho_part:+.3f}, {attenuation_pct:+.1f}% change)")

Spearman(|cos|, Q):                    rho=+0.153,  p=0.374
  → not significant at this sample size

Spearman(sem_sim, Q):                  rho=-0.092,  p=0.593
  → not significant at this sample size

Pearson r(|cos|, sem_sim):             r=+0.528,   p=0.001
  → significant (p=0.001) — covariates correlated, partial Spearman needed

Partial Spearman(|cos|, Q | sem_sim):  rho=+0.236,  p=0.166
  → not significant at this sample size after partialling out sem_sim
  Attenuation: raw rho=+0.153 → partial rho=+0.236  (-54.5% change)


### Q ~ $|\cos|$ + sem\_sim — multiple linear regression

Parametric complement to the partial Spearman above: the OLS partial slope of $|\cos|$ controlling `sem_sim`, given the **same bootstrap-CI + permutation-p treatment** used for the logistic models. We use resampling-based inference (not the parametric SEs) because `Q` is clipped to $[-2, 2]$, which violates the OLS error assumptions. Predictors are standardized so the two $\beta$'s are directly comparable; **Spearman remains the robust primary test**.

In [ ]:
from sklearn.linear_model import LinearRegression

def bootstrap_perm_ols(X, y, feat_names):
    """OLS inference mirroring bootstrap_perm: in-sample R^2 + standardized
    coefficients, LOO coefficient sign stability, bootstrap 95% CIs, and a
    permutation p-value on R^2 (resampling-based, since Q is clipped)."""
    n            = len(y)
    base         = LinearRegression().fit(X, y)
    r2_obs       = base.score(X, y)
    coefs_obs    = base.coef_

    # --- LOO coefficient sign stability ---
    loo = np.array([LinearRegression().fit(np.delete(X, i, axis=0),
                                            np.delete(y, i)).coef_
                    for i in range(n)])
    for j, nm in enumerate(feat_names):
        col   = loo[:, j]
        flips = int((np.sign(col) != np.sign(coefs_obs[j])).sum())
        print(f"  LOO beta[{nm}]: min={col.min():+.3f} max={col.max():+.3f}  "
              f"sign_flips={flips}/{n}")

    # --- Bootstrap 95% CI on R^2 and coefficients ---
    b_r2, b_co = [], []
    for _ in range(N_BOOT):
        idx = rng.integers(0, n, n)
        m   = LinearRegression().fit(X[idx], y[idx])
        b_r2.append(m.score(X[idx], y[idx]))
        b_co.append(m.coef_)
    b_r2, b_co  = np.array(b_r2), np.array(b_co)
    r2_lo, r2_hi = np.percentile(b_r2, [2.5, 97.5])
    print(f"R^2 = {r2_obs:.3f}  95% CI [{r2_lo:.3f}, {r2_hi:.3f}]")
    for j, nm in enumerate(feat_names):
        lo, hi = np.percentile(b_co[:, j], [2.5, 97.5])
        excl   = "excludes 0" if (lo > 0 or hi < 0) else "includes 0"
        print(f"  beta[{nm}] = {coefs_obs[j]:+.3f}  95% CI [{lo:+.3f}, {hi:+.3f}]  ({excl})")

    # --- Permutation p on R^2 (shuffle Q) ---
    perm = np.empty(N_PERM)
    for k in range(N_PERM):
        yk      = rng.permutation(y)
        perm[k] = LinearRegression().fit(X, yk).score(X, yk)
    p_r2 = (np.sum(perm >= r2_obs) + 1) / (N_PERM + 1)
    print(f"permutation p (R^2, one-sided) = {p_r2:.4f}")

    return {
        "r2":        float(r2_obs),
        "r2_ci_lo":  float(r2_lo),
        "r2_ci_hi":  float(r2_hi),
        "perm_p_r2": float(p_r2),
        **{f"beta_{nm}": float(coefs_obs[j]) for j, nm in enumerate(feat_names)},
    }

In [ ]:
# Q ~ |cos| + sem_sim   (standardized predictors so the two betas are comparable)
X_q   = StandardScaler().fit_transform(df[['cosine_abs', 'sem_sim']].to_numpy())
y_q   = df['Q'].to_numpy()
feat  = ['|cos|', 'sem_sim']

ols = LinearRegression().fit(X_q, y_q)
print(f"Q ~ |cos| + sem_sim   (standardized predictors, n={len(y_q)})")
print(f"in-sample R^2 = {ols.score(X_q, y_q):.3f}")
print(f"  beta[|cos|]   = {ols.coef_[0]:+.3f}")
print(f"  beta[sem_sim] = {ols.coef_[1]:+.3f}\n")

ols_q_stats = bootstrap_perm_ols(X_q, y_q, feat)

## Save RQ1 summary

Merge the Spearman/Pearson results with the logistic-regression table and write `results/rq1_summary.csv`.

In [133]:
out_dir = REPO_ROOT / "results"
out_dir.mkdir(parents=True, exist_ok=True)

spearman_rows = [
    {"test_type": "spearman",         "test": "cos_Q",           "stat": float(rho_cq),   "p": float(p_cq),   "n": n},
    {"test_type": "pearson_perm",     "test": "cos_Q",           "stat": float(r_cq),     "p": float(p_cq_perm), "n": n},
    {"test_type": "spearman",         "test": "signedcos_Q",     "stat": float(rho_sc),   "p": float(p_sc),     "n": n},
    {"test_type": "pearson_perm",     "test": "signedcos_Q",     "stat": float(r_sc),     "p": float(p_sc_perm), "n": n},
    {"test_type": "spearman",         "test": "sem_Q",           "stat": float(rho_sq),   "p": float(p_sq),   "n": n},
    {"test_type": "pearson_perm",     "test": "sem_Q",           "stat": float(r_sq),     "p": float(p_sq_perm), "n": n},
    {"test_type": "spearman_partial", "test": "cos_Q_given_sem", "stat": float(rho_part), "p": float(p_part), "n": n,
     "attenuation_pct": round(float(attenuation_pct), 1)},
    {"test_type": "pearson",          "test": "cos_sem",         "stat": float(r_cs),     "p": float(p_cs),   "n": n},
]
spearman_df_out = pd.DataFrame(spearman_rows)

lr_cols = ["experiment", "outcome", "predictors", "n_features", "auc_insample", "auc_loo", "ci_lo", "ci_hi", "perm_p"]
lr_rows = results_df[lr_cols].copy()
lr_rows.insert(0, "test_type", "logistic_lr")
lr_rows.rename(columns={"experiment": "test", "auc_insample": "stat"}, inplace=True)

summary = pd.concat([spearman_df_out, lr_rows], ignore_index=True, sort=False)
out_path = out_dir / "rq1_summary.csv"
summary.to_csv(out_path, index=False)

print(f"Saved {len(summary)} rows → {out_path}")
print(summary[["test_type", "test", "stat", "p"]].to_string(index=False))

Saved 11 rows → /Users/federicoscaffidimuta/Desktop/Third year/ML project/steering-vector-composition/results/rq1_summary.csv
       test_type             test      stat        p
        spearman            cos_Q  0.152638 0.374147
        spearman            sem_Q -0.092149 0.592980
spearman_partial  cos_Q_given_sem  0.235779 0.166250
         pearson          cos_sem  0.528380 0.000925
     logistic_lr        1_abs_cos  0.828283      NaN
     logistic_lr        2_sem_sim  0.646465      NaN
     logistic_lr     3_signed_cos  0.676768      NaN
     logistic_lr  4_multi_abs_cos  0.698283      NaN
     logistic_lr    5_abs_cos_sem  0.818182      NaN
     logistic_lr 6_signed_cos_sem  0.666667      NaN
     logistic_lr  7_multi_abs_sem  0.713833      NaN


In [134]:
print(df[df['is_additive'] == 1][['trait_a', 'trait_b', 'cos', 'cosine_abs', 'sem_sim', 'cosine_abs']].sort_values('cos'))
print("\nAdditive pairs — cos stats:")
print(df[df['is_additive'] == 1]['cos'].describe())
print("\nNon-additive pairs — cos stats:")
print(df[df['is_additive'] == 0]['cos'].describe())

      trait_a        trait_b     cos  cosine_abs   sem_sim  cosine_abs
6   apathetic  power_seeking  0.0063      0.0063  0.291267      0.0063
31   humorous  power_seeking  0.0727      0.0727  0.190971      0.0727
24  formality  power_seeking  0.1832      0.1832  0.235060      0.1832

Additive pairs — cos stats:
count    3.000000
mean     0.087400
std      0.089361
min      0.006300
25%      0.039500
50%      0.072700
75%      0.127950
max      0.183200
Name: cos, dtype: float64

Non-additive pairs — cos stats:
count    33.000000
mean      0.168179
std       0.240858
min      -0.522500
25%       0.015000
50%       0.226200
75%       0.302600
max       0.694800
Name: cos, dtype: float64


In [135]:
df['has_halluc'] = (df['trait_a']=='hallucinating') | (df['trait_b']=='hallucinating')
print(pd.crosstab(df['has_halluc'], df['is_additive']))


is_additive   0  1
has_halluc        
False        25  3
True          8  0
